In [5]:
from pathlib import Path

from datasets import DatasetDict

from src.config.paths import DEFAULT_TRAIN_LLM_CONFIG_PATH
from src.ml.config.loader import load_experiment_config, load_llm_training_config
from src.ml.datasets.splits import Splits, build_splits
from src.ml.llm.dataset import (
    SPLIT_NAMES,
    build_llm_dataset,
    build_split_dataset,
    push_to_hub,
    summarize,
)
from src.ml.llm.finetune import (
    preflight,
    resolve_run_name,
    run_training,
)

In [2]:
LLM_CONFIG = "llama31_8b_qlora"
config_path = f"configs/train_llm/{LLM_CONFIG}.yaml"

resolved = Path(config_path) if config_path else DEFAULT_TRAIN_LLM_CONFIG_PATH
config = load_llm_training_config(resolved)

In [4]:
max_train_samples = 10
if max_train_samples is not None:
    config.data.max_train_samples = max_train_samples

In [6]:
run_name = None
resolved_run_name = resolve_run_name(config, run_name)

In [ ]:
resolved_run_name

'nba-bets-2026-09-28_19.09.23'

In [11]:
from src.config.secrets import (
    GLOBAL_SECRETS_FILE,
    HF_TOKEN_ENV_VAR,
    WANDB_TOKEN_ENV_VAR,
    require_env,
)


In [12]:
HF_TOKEN_ENV_VAR

'HF_llm_training_token'

In [24]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Expand path to the .env file
env_path = Path.expanduser(Path("~/.secrets/nba_bets.env"))

# Load environment variables from the specific path
load_dotenv(dotenv_path=env_path)

# Access the variable
token = os.getenv("HF_llm_training_token")


In [25]:
from huggingface_hub import login

login(token, add_to_git_credential=True)

In [30]:
from src.ml.llm.finetune import build_base_model, build_tokenizer, load_train_dataset

In [ ]:
train = load_train_dataset(config)

README.md:   0%|          | 0.00/569 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.65M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/842k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/299k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/23247 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1018 [00:00<?, ? examples/s]

In [28]:
tokenizer = build_tokenizer(config)

In [29]:
tokenizer

PreTrainedTokenizerFast(name_or_path='meta-llama/Meta-Llama-3.1-8B', vocab_size=128000, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>', 'pad_token': '<|end_of_text|>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128005: AddedToken(

In [31]:
base_model = build_base_model(config, tokenizer)

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`